# **MOBILE ROBOTS Project Report**
**Group 38**

| **Name** | **SCIPER Number** | **Email** |
|---------|--------------------------|-----------|
| Killian Baillifard | 393835 | killian.baillifard@epfl.ch |
| Kael Murphy | 413744 | kael.murphy@epfl.ch |
| Moritz Tschudin | 327569 | moritz.tschudin@epfl.ch |
| Alex Gochely | 361135 | alex.gochely@epfl.ch |

### **Table of Contents**

1. [Introduction](#introduction)
2. [Environment](#environment)
3. [Vision](#vision)
4. [Global Navigation](#global-navigation)
5. [Motion Control](#motion-control)
6. [Local Navigation](#local-navigation)
7. [Filtering](#filtering)
8. [GUI](#gui-dashboard-and-demo)
9. [X](#XY)
10. [Conclusion](#conclusion)

## Introduction
The project for the course Basics of Mobile Robotics (BOMR) focuses on navigating the Thymio robot through an environment containing global obstacles, local obstacles, a defined start, and a goal. The main objective is to enable the robot to extract a global map from an overhead vision system, compute an (optimal) global path, and follow it autonomously. During navigation, the Thymio must also react to unexpected local obstacles placed in its path and handle special conditions such as kidnapping or losing access to the camera used for pose correction.

The project is structured into five modules, as defined in the official project description:

- **Vision** treated by *Kael*  
- **Global Navigation** treated by *Moritz*  
- **Motion Control** and **Local Navigation** treated by *Killian*  
- **Filtering** (Bayesian pose estimation) treated by *Alex*

## Environment

For our environment, we imagined the following scenario:  
Since last summer, extensive and noisy construction work has been taking place in the heart of the EPFL campus, on the Esplanade. These operations involve heavy vehicles such as excavators and trucks, as well as piles of rubble. We imagined a group of EPFL researchers aiming to deploy a mobile robot capable of autonomously delivering material across the construction site, an area where robotics and automation are still under-explored.

Because a construction site is a highly dynamic environment, the robot must be adaptable and capable of re-planning an optimal path every day, or even every hour, depending on changes in the environment. A drone would ideally provide an overhead view of the current state of the site, in our project, this drone is simulated by a camera mounted on the ceiling.

To simplify the scenario into a proof-of-concept using the Thymio robot, we abstract construction elements (trucks, excavators, rubble piles) into convex polygonal obstacles. Local obstacles, objects not visible to the camera or not included in the global map,  are represented by white cylinders topped with construction helmets, symbolizing construction workers who may unpredictably step into the robot’s path. These require the Thymio to perform local obstacle avoidance in real time. This setup is illustrated in Figure 01.

<p align="center">
  <img src="images/BOMR.png" width="90%">
</p>

**Figure 01:**
Real-world construction site (A), corresponding global obstacles such as vehicles and rubble (B), and the resulting simplified polygons and map used for creating the environment (C). *Source: See links on image.*


The final environment therefore consists of:
- 4 ArUco markers defining the global frame  
- A white paper sheet including a zone(1250 × 740 mm) representing the workspace  
- 4 convex polygonal obstacles (global, static)  
- A start and goal, each represented by an ArUco marker  
- Local obstacles, represented by white cylinders with helmets  
This is illusrtated in *Figure 02*.
<p align="center">
  <img src="images/Setup_real_life.png" width="80%">
</p>

**Figure 02:** 
Showing real map (B) and close-up picture of the obstacles in (A).

#### Use of AI Tools
Generative AI tools such as the BOMR AI Tutor, ChatGPT 5.1, and GitHub Copilot were used as supportive resources during the project. They were used for improving code structure, readability, commenting, debugging assistance, and refining the written text of the report. Their use was combined with the official course material of MICRO-452 Basics of Mobile Robotics and its Jupyter notebooks, which remained the primary reference for all technical decisions and implementations. Any additional external sources are cited directly in the report.

Next, we provide a detailed description of each of the five project modules.



## Global Navigation

In this section, we describe the design and implementation of the global navigation module. Its purpose is to compute a collision-free geometrically optimal path for the Thymio robot, based on the environment reconstructed by the Vision Module.

Our approach uses a Visibility Graph combined with A* search, as introduced in the course. Obstacles detected in the Esplanade construction scenario (e.g., excavators, transport vehicles, gravel piles) are abstracted as convex polygons, allowing efficient geometric reasoning.

We use the [Shapely library](https://shapely.readthedocs.io/en/stable/)
for computational geometry (e.g., `Polygon`, `LineString`, intersection tests), which greatly simplifies handling polygonal obstacles.

### Chosen Planning Approach: Visibility Graph

##### Advantages
- Produces the geometrically optimal path in continuous space.
- Requires far fewer nodes compared to grid-based planning.
- Generates a compact list of waypoints, suitable for robot execution.
- Works especially well with convex polygonal obstacles.

##### Limitations
- Requires precise polygon vertices from the Vision Module.
- Robot modeled as a point requires obstacle inflation to avoid collisions.
- Path must be recomputed when obstacles move.

### Overview of the Planning Pipeline

The global planner consists of the following stages:

1. Input from Vision Module
2. Polygon orientation normalization (CCW)
3. Construct obstacle polygons
4. Inflate polygons by ε
5. Extract nodes (vertices + start + goal)
6. Build the Visibility Graph
7. Run A* search
8. Return final waypoint list

---

### 1. Input from the Vision Module

The Vision Module provides obstacle vertices as well as start and goal coordinates in the format:

`type, id, label, x, y`

Example:

```
poly1, poly1, A, 123.4, 567.8
poly1, poly1, B, 200.0, 540.0
start, start, S,  50.0, 900.0
goal,  goal,  G, 900.0,  60.0
```

These coordinates are already converted into the world coordinate system in millimeters using camera calibration.

---

### 2. Polygon Orientation (CCW)

Shapely requires polygons to have consistent vertex ordering.
We ensure counter-clockwise (CCW) orientation using the signed area (shoelace) method:

\begin{array}{l}
A = \frac{1}{2} \sum_{i=1}^{n} \left( x_i y_{i+1} - x_{i+1} y_i \right)
\end{array}

- If \( A > 0 \) → polygon is CCW  
- If \( A < 0 \) → reverse vertex order  

All polygons are therefore normalized to CCW orientation.

Short helpers: CCW + small inflated polygon snippet
```python
# Short helpers: shoelace area and enforce counter-clockwise
def signed_area(ring):
    a = 0.0
    for (x1,y1), (x2,y2) in zip(ring, ring[1:]+ring[:1]):
        a += x1*y2 - x2*y1
    return 0.5*a

def ensure_ccw(ring):
    return ring if signed_area(ring) > 0 else list(reversed(ring))
```

Source: [The Shoelace Algorithm](https://www.101computing.net/the-shoelace-algorithm/).

---

### 3. Constructing Polygons

We reconstruct each obstacle as a Shapely `Polygon`.

This allows:

- robust collision detection
- buffering / inflation
- geometric operations
- clean visualization

We also render the reconstructed map for debugging and verification.

<p align="center">
  <img src="images/polygons_0.png" width="50%">
</p>

---

### 4. Obstacle Inflation

To avoid collisions, obstacles are expanded in configuration space by an inflation parameter ε (robot safety margin).

Shapely's buffer operation approximates Minkowski sum:

\begin{array}{l}
\tilde{\mathcal{O}} = \mathcal{O} \oplus B(\varepsilon)
\end{array}

We use:

```
poly.buffer(epsilon, join_style=2, resolution=1)
```

which preserves sharp edges and limits vertex growth.

The inflated polygons create the configuration-space map used for planning. 

<p align="center">
  <img src="images/polygons_4.png" width="100%">
</p>

Minimal inflate_polygons demonstration:

```python
from shapely.geometry import Polygon

def inflate_ring(ring, eps_mm=20.0, join_style=2, resolution=1):
    """Return buffered exterior ring coordinates (preserve sharp corners)."""
    poly = Polygon(ring)
    buf = poly.buffer(eps_mm, join_style=join_style, resolution=resolution)
    if buf.geom_type == "MultiPolygon":
        # choose the largest part (rare)
        buf = max(buf.geoms, key=lambda p: p.area)
    coords = list(buf.exterior.coords)[:-1]  # drop closing vertex
    return ensure_ccw(list(coords))
```

Source: [Wikipedia](https://en.wikipedia.org/wiki/Minkowski_addition)

---

### 5. Node Extraction

We define nodes as:

- all vertices of inflated obstacles except the outer boundary poly0
- the start coordinate
- the goal coordinate

These are stored as:
```
node_coords = [(x0, y0), ...]
node_labels = ["poly1_0", ..., "start", "goal"]
```

The outer map boundary is excluded since its vertices are not used for visibility edges.

---

### 6. Building the Visibility Graph

For every pair of nodes `(i,j)`, we test whether the segment connecting them is obstacle-free.
We rely on Shapely:

- `LineString(segment).crosses(polygon)`
- `LineString(segment).within(polygon)`
- bounding-box prechecks

We allow segment–obstacle tangency (parallel to edges) since obstacles are already inflated.

Helper functions:

- `segment_visibile()`
- `euclidean_distance(i, j)`
- `build_neighbors(...)`

The adjacency list defines the Visibility Graph.

Example neighbor count:
```
Node 12 (poly3_0) has 18 neighbors
Node 212 (start) has 55 neighbors
Node 213 (goal) has 30 neighbors
```

A visualization confirms correct visibility edge construction.

<p align="center">
  <img src="images/polygons_1.png" width="50%">
</p>

Short segment visibility & neighbor builder:

```python
from shapely.geometry import LineString

def segment_visible(p, q, world_poly, obstacle_polygons):
    seg = LineString([p,q])
    # Must remain inside the world
    if not (seg.within(world_poly) or seg.touches(world_poly)):
        return False
    # Disallow segments that cross obstacle interiors
    for obs in obstacle_polygons:
        if seg.crosses(obs) or seg.within(obs) or obs.contains(seg):
            return False
    return True

def build_visibility(node_coords, world_poly, obstacles):
    N = len(node_coords)
    neighbors = [[] for _ in range(N)]
    for i in range(N):
        for j in range(i+1,N):
            if segment_visible(node_coords[i], node_coords[j], world_poly, obstacles):
                d = ((node_coords[i][0]-node_coords[j][0])**2 + (node_coords[i][1]-node_coords[j][1])**2)**0.5
                neighbors[i].append((j,d)); neighbors[j].append((i,d))
    return neighbors
```

---

### 7. A* Path Planning

We apply A* search on the visibility graph.
The cost function is:

\begin{array}{ll}
f(n)=g(n)+h(n)
\end{array}

Where:

- $g(n)$: accumulated distance from start  
- $h(n) = \| n - \text{goal} \|_2$: Euclidean distance heuristic

The heuristic is admissible and consistent, ensuring optimality.

Although turn penalties can be included, the visibility graph already contains few nodes and produces nearly straight-line paths, making additional penalties less necessary.

<p align="center">
  <img src="images/polygons_2.png" width="50%">
</p>

Minimal A* snippet:

```python
import heapq, math
def astar_short(neighbors, coords, start_idx, goal_idx):
    g = {start_idx: 0.0}
    parents = {}
    openq = [(math.hypot(coords[start_idx][0]-coords[goal_idx][0],
                          coords[start_idx][1]-coords[goal_idx][1]), start_idx)]
    closed = set()
    while openq:
        _, cur = heapq.heappop(openq)
        if cur in closed: continue
        closed.add(cur)
        if cur == goal_idx: break
        for (nbr, w) in neighbors[cur]:
            newg = g[cur] + w
            if newg < g.get(nbr, float('inf')):
                g[nbr] = newg
                parents[nbr] = cur
                h = math.hypot(coords[nbr][0]-coords[goal_idx][0], coords[nbr][1]-coords[goal_idx][1])
                heapq.heappush(openq, (newg+h, nbr))
    if goal_idx not in parents and start_idx != goal_idx: return None
    path = [goal_idx]
    while path[-1] != start_idx:
        path.append(parents[path[-1]])
    return list(reversed(path))
```

---

### 8. Output and Integration with the Control Module

Running A* yields a waypoint sequence:
```
[(10.0, 900.0), (75.3, 590.1), (595.6, 176.6), (900.0, 10.0)]
```

This path is sent to the Control Module as the global plan.

Below is an example comparing the optimal A* path (red), the measured odometry of the robot (blue), and the inflated polygonal environment:

<p align="center">
  <img src="images/polygons_3.png" width="50%">
</p>


#### Note about odometry

The Thymio successfully followed the computed trajectory.

Observations:

- The odometry trace follows the theoretical path closely.  
- Expected drift occurs, consistent with dead-reckoning.  
- A brief wheel slip caused a small jump in the odometry trace.  
- This confirms that the encoder-based odometry and coordinate transforms were implemented correctly.  
- Accurate localization requires vision-based pose estimation, integrated later.

---

### Implementation Summary & File Structure

The global navigation module is implemented across two files:

#### `globalnav.py` — core planning pipeline

Implements:

- input parsing  
- CCW normalization  
- polygon construction (Shapely)  
- obstacle inflation  
- node extraction  
- visibility graph construction  
- A* path computation  

Exposed functions:

- `compute_global_path(map_array, epsilon_mm)` → returns the final waypoint list  
- `compute_global_path_with_debug(map_array, epsilon_mm)` → returns the path and all intermediate data (polygons, inflated polygons, nodes, neighbors, edges, path indices) for plotting  

#### `globalnav_plot.py` — visualization helper

- Calls `compute_global_path_with_debug()` once  
- Draws inflated polygons, edges, A* path, nodes, legends  
- Returns matplotlib handles for efficient updating  
- Ensures ε is displayed consistently in the legend  

---

### Call of A* Once

During dashboard setup, `setup_globalnav_plot(...)` is called **once**.  
Inside this function:

- `compute_global_path_with_debug()` is executed  
- The computed path and debug data are stored  
- The robot control logic reuses this exact path  

Consequences:

- No duplicate A* calls → avoiding inconsistent paths  
- The inflation value ε is identical for:
  - the computed path  
  - the displayed configuration-space map  
  - the robot’s executed trajectory  
- Ensures perfect alignment between:
  - the displayed map  
  - the A* trajectory  
  - the robot’s real motion  

This prevents subtle bugs such as mismatched inflation radii or inconsistent path computation.

### Parameters, tuning and how we chose ε (epsilon)

This planner uses a few parameters that directly affect correctness, safety, and runtime. We explain the important ones here, why they matter, and how we chose them empirically.

- ε (epsilon, inflation radius) — purpose: model the robot as a point in configuration space by expanding obstacles so that planning avoids robot body collisions. In `globalnav.py` and `globalnav_plot.py` this value is passed as `epsilon_mm` (units: mm). Default examples in the code/demo use values in the range 20–90 mm (the planner default is 20 mm; `demo.py` uses `GLOB_NAV_EPSILON = 90`).

  Why it matters:
  - Small ε → narrow clearance, paths may pass too close to obstacles and cause collisions on a real robot.
  - Large ε → obstacles may merge or block corridors and prevent any valid path from existing.


- `poly.buffer(..., join_style=2, resolution=1)` — buffer settings in `inflate_polygons`:
  - `join_style=2` (mitre) preserves sharp corners instead of rounding them, which makes the inflated polygon closer to the true Minkowski sum for polygonal obstacles. This matters for visibility since vertex positions remain meaningful.  
  - `resolution=1` controls the number of points used to approximate rounded corners — `1` is coarse (few points) so it's faster and produces fewer vertices; increase it for more fidelity at the cost of runtime.

- Visibility graph construction choices:
  - Nodes are the vertices of the *inflated* obstacles (except the outer `poly0`) plus `start` and `goal`. Using inflated vertices reduces false negatives in visibility and keeps the plan away from obstacles.
  - Segment policy: we *allow* tangency (touching) to obstacles but reject segments that cross obstacle interiors or lie strictly within them. This policy + inflation gives the planner a conservative, collision-avoiding behaviour.

- A* settings and heuristic:
  - Edge weights are Euclidean distances and the heuristic is Euclidean distance to the goal: admissible and consistent → A* finds an optimal path in geometric length.
  - We did not add turn penalties in the current implementation. Turn-penalties (e.g., penalizing large heading changes)

## Motion control

### Strategy

Controlling the robot require some kind of closed loop control, whether it's a P controller, an Astolfi or one of another kind. This loop could be implemented on the PC with Python, but this would induce a **huge latency** in the feedback loop, which would easily make the system **unstable**. That's why we prefered to implement the closed loop control law directly on the **Thymio** in **Aseba** to be aware of the features of the language. But this requires synchronization between the PC and the robot.

#### Initialization

A **Thymio** class handle this synchronization. It first loads the **Aseba** program and do a preprocessing step to load some constants into the program.

```python
class Thymio():

    def __init__(self, ...) -> None:
        
        # Init code
        ...

        # Calibration and program
        self.cal = calibration
        with open('thymio.aesl') as file:
            self.program = file.read().format(
                X0      = self.x,
                Y0      = self.y,
                X1      = int(np.round(x1)),
                Y1      = int(np.round(y1)),
                THETA0  = rad_to_lsb(self.theta),
                K_D     = self.cal.kD,
                K_THETA = self.cal.kTheta
            )
```

These **initialization constants** look like follow in the Thymio program.

```aseba
# Pose
var x_mm = {X0}
var x_um = 0
var y_mm = {Y0}
var y_um = 0
var theta = {THETA0}

# Reference
var r_x_mm = {X1}
var r_y_mm = {Y1}
```

Then a **context manager** handle connection and disconnection from the robot when the python code enter or exit a **with** statement :

```python
def __enter__(self) -> 'Thymio':

        # Initialize client
        self.client.__enter__()
        self.client.add_event_received_listener(self.on_event_received)
        
        # Connect to node
        self.node = aw(self.client.lock())
        ...
        
        # Compile program
        error = aw(self.node.compile(self.program))
        ...

        # Start program
        error = aw(self.node.run())
        ...
```

#### Communication

Communications from the **PC to the robot** are done by simply **writing variables** of the program. Below is an exemple for setting Thymios position and setpoint.

```python
def set_pose(self, x: float, y: float, theta: float) -> None:
    self.x = int(np.round(x))
    self.y = int(np.round(y))
    self.theta = theta
    aw(self.node.set_variables({'x_mm': [self.x], 'y_mm': [self.y], 'theta': [rad_to_lsb(theta)]}))

def set_target(self, x: float, y: float) -> None:
    self.blocked = False
    x = int(np.round(x))
    y = int(np.round(y))
    aw(self.node.set_variables({'state': [0], 'r_x_mm': [x], 'r_y_mm': [y]}))
```

Then communications from the **robot to the PC** is done by **emitting events** with data periodically. Below is an exemple to return odometry data.

```aseba
# Emit pose
emit pos [s, ms, x_mm, y_mm, theta, v, omega]
```

Then the **event is caught** in the thymio object :

```python
def on_event_received(self, node, event_name, event_data):

    match event_name:
        case 'pos':
            ...
        case 'blocked':
            ...
        case 'clear':
            ...
```

### Odometry

#### Pose

The Thymio robot in our use case has **3 DOF**, two translational and one rotational : $[x, y, \theta]^T$. The goal is to find the change in position over one time step :
$$
\begin{equation}
\begin{bmatrix}
    x_+ \\
    y_+ \\
    \theta_+
\end{bmatrix} =
\begin{bmatrix}
    x \\
    y \\
    \theta
\end{bmatrix} +
\begin{bmatrix}
    \Delta x \\
    \Delta y \\
    \Delta \theta
\end{bmatrix}
\end{equation}
$$

#### Change in heading

When moving one time step, the robot advance by $\Delta d_L$ and $\Delta d_R$ on each wheel. The track width $w$ being constant, we can consider that they are two arc length of two concentric circles of radiuses $w + r$ and $r$.

<p align="center">
    <img src="images/local-nav-odometry-heading.png" alt="odometry-heading" width="200"/>
</p>

With $\Delta d_L$ is the interior arc and $\Delta d_R$ the exterior arc, from the definition of the radian $\alpha = d / r$, we can solve the increment in angle that $\Delta d_L$ and $\Delta d_R$ describe :
$$
\begin{align}
\Delta \theta = \frac{\Delta d_L}{r} &= \frac{\Delta d_R}{w + r} \\
\Rightarrow \frac{r}{\Delta d_L} &= \frac{w + r}{\Delta d_R} \\
\Rightarrow r &= \frac{w \cdot \Delta d_L}{\Delta d_R - \Delta d_L} \\
\Rightarrow \Delta \theta &= \frac{\Delta d_R - \Delta d_L}{w}
\end{align}
$$

#### Change in position

To integrate the position, we could consider the same geometrical setup as before, but it would be a bit computationally heavy, and would lead to special cases (i.e. radius at infinity when moving in a straight line). The **mid-point rule** also known as **Runge–Kutta 2** (**RK2**) is used instead.

<p align="center">
    <img src="images/local-nav-odometry-position.png" alt="odometry-position" width="150"/>
</p>

We consider that change of coordinate has followed a straight line $\Delta s$ at the mid-point heading $\theta_{mid}$.

$$
\begin{align}
\Delta s &= \frac{\Delta d_L + \Delta d_R}{2} \\
\theta_{mid} &= \theta + \frac{\Delta \theta}{2} \\
\Delta x &= \Delta s \cdot \cos \theta_{mid} \\
\Delta y &= \Delta s \cdot \sin \theta_{mid} \\
\end{align}
$$

#### Change in wheel distance

First, we solve the simple case of linear motion for each wheel. From the Thymio cheat sheet we get the constants :
$$
\begin{equation}
\begin{aligned}
\Delta t &= \frac{1}{100 \text{ Hz}} = \frac{10}{1000} \text{ s} \\
v_{mm/s} &= 20 \text{ cm/s} = 200 \text{ mm/s} \\
v_{lsb/s} &= 500 \text{ lsb/s}
\end{aligned}
\end{equation}
$$

To have a large enough full scale range of $\approx \pm 2^{15} = \pm 32'768 \approx \pm 32 \text{ m}$, the initial increment estimation is computed in $\text{mm}$ :
$$
\begin{align}
\Delta d_{mm} &= v_{lsb/s} \cdot \frac{v_{mm/s}}{v_{lsb/s}} \cdot \Delta t \\
&= v_{lsb/s} \cdot \frac{200}{500} \cdot \frac{10}{1000} \\
&= v_{lsb/s} \cdot \frac{2}{500} \\
\end{align}
$$
However, because of fixed point arithmetic, only the speeds $250$ and $500$ can be differentiated. They will respectively give increments of $\pm 1$ and $\pm 2$ on the position estimation, so almost $8$ bits of precision are lost. To take into acount those lost $8$ bits, we must estimate the increment in $\mu m$ :
$$
\begin{equation}
\Delta d_{\mu m} = 1000 \cdot \Delta d_{mm} = v_{lsb/s} \cdot \frac{20}{5} = v_{lsb/s} \cdot 4
\end{equation}
$$

After experimentation and calibration, it turns out that the given speeds in $mm/s$ and $lsb/s$ are not exact. The calibration constant found by trial and error is :
$$
\begin{align}
\Delta d_{\mu m} = v_{lsb/s} \cdot 3.1254 \\
k_d = 3.1254 \approx \frac{31'254}{10'000}
\end{align}
$$

Which give a maximum speed closer to $15.6 \text{ cm/s}$.

In aseba, this is then implemented like below :

```aseba
call math.muldiv(d_l_um, motor.left.speed, {K_D}, 10000)
call math.muldiv(d_r_um, motor.right.speed, {K_D}, 10000)
```

With **K_D** set to $31254$ and `math.muldiv` computing the multiplication in a 32 bit register to avoid a 16 bit overflow during the multiplication.

#### Change in robot heading

We computed earlier that we need to find the following angle increment :
$$
\begin{equation}
\Delta \theta = \frac{\Delta d_R - \Delta d_L}{w} \, rad
\end{equation}
$$

On the Thymio, angles use a fixed point representation in the range $[-\pi, \pi[$ mapped to $[-2^{15}, 2^{15}[$. So one radian is equal to :
$$
1 \, rad = \frac{2^{15}}{\pi} \approx 10'430
$$

If we try to deduce a constant with Thymio track width of $w = 95'000 \, \mu m$ to compute angles in these units, we get :
$$
\begin{align}
\Delta \theta &= \frac{\Delta d_R - \Delta d_L}{w} \, rad = (\Delta d_R - \Delta d_L) \cdot \frac{2^{15}}{\pi \cdot w} \\
k_\theta &= \frac{2^{15}}{\pi \cdot w} \approx \frac{10'430}{95'000} \approx \frac{1'098}{10'000}
\end{align}
$$

In aseba, this is the implemented like below :
```aseba
call math.muldiv(d_theta, d_r_um - d_l_um, {K_THETA}, 10000)
```

With **K_THETA** set to $1098$.

### Controller

#### Astolfi

The pose estimation is perfectly suited for an Astolfi controller, with $x_e$, $y_e$ and $\theta_e$ the pose error, and the change of coordiates :
$$
\begin{align}
\rho &= \sqrt{x_e^2 + y_e^2} \\
\alpha &= \text{atan2}(y_e, x_e) - \theta \\
\beta &= \theta_e - \alpha
\end{align}
$$

With $\rho$ is the distance to the target, $\alpha$ the target heading error, and $\beta$ the end heading error. The control law is :
$$
\begin{align}
v &= k_\rho \cdot \rho \\
\omega &= k_\alpha \cdot \alpha + k_\beta \cdot \beta \\
k_\rho &> 0, k_\beta < 0, k_\alpha - k_\rho > 0
\end{align}
$$

This controller works fine, but it tends to move in big arcs before getting to the target, which is not optimal in a tight environnement.

#### Tweaked astolfi

To fix this the controller must be changed a little. The end heading error $\beta$ is not needed, so it is removed. Then, to make clear that this is a different controller, $\rho$ is renamed as $e$ and $\alpha$ becomes $\varepsilon$ :
$$
\begin{align}
e &= \sqrt{x_e^2 + y_e^2} \\
\varepsilon &= \text{atan2}(y_e, x_e) - \theta
\end{align}
$$

Then, assuming $\varepsilon \in [-\pi, \pi[$, the controller is modified as :
$$
\begin{align}
v &= k_e \cdot e \cdot \left(\pi - |\varepsilon|\right) \\
\omega &= k_\varepsilon \cdot \varepsilon
\end{align}
$$

This **prevents the robot from moving** forward **until it's headed in the right direction**. Then the robot will move at **constant speed** before reaching $e_{max}$, from which it will act as a classic P controller. The speed PI controller will take care of the **soft start** while the position P controller does the **soft stop**. Implementation in Aseba looks like follow :

```aseba
# Compute carthesian error
e_x_mm = r_x_mm - x_mm
e_y_mm = r_y_mm - y_mm

# Compute polar error, clamp to avoid L2 norm overflow
call math.clamp(e_x_mm, e_x_mm, -127, 127)
call math.clamp(e_y_mm, e_y_mm, -127, 127)
call math.sqrt(e_mm, (e_x_mm * e_x_mm) + (e_y_mm * e_y_mm))
call math.atan2(epsilon, e_y_mm, e_x_mm)
epsilon -= theta

# Compute control outputs
call math.muldiv(v, e_mm, k_e_num, k_e_den)
call math.muldiv(v, v, 32767 - abs epsilon, 32767)
call math.muldiv(omega, epsilon, k_epsilon_num, k_epsilon_den)

# Clamp linear speed
call math.clamp(v, v, -200, 200)
left = v - omega
right = v + omega
```


## Local navigation

### Strategy

The custom controller follows the path **one waypoint at a time**.

<p align="center">
    <img src="images/local-nav-path-following.png" alt="path-following" width="200"/>
</p>

When detecting the obstacle with the horizontal proximity sensors, the avoidance trajectory should be pushed away from the obstacle, **tangeant** to it. This approach is simillar to a **potential field** repulsing the path.

<p align="center">
    <img src="images/local-nav-potential-field.png" alt="potential-field" width="400"/>
</p>

Knowing the path was generated with a visibility graph, we can assume each vertex is close to an exclusion zone. The avoidance should hence be done on the **path exterior**, while considering that the robot is not perfectly on the path. It must intersect its trajectory to the next waypoint or the one after to continue :

<p align="center">
    <img src="images/local-nav-avoidance-strategy.png" alt="avoidance-strategy" width="400"/>
</p>



### State machine

The proximity sensors being far ahead of the rotation center of the robot, directly implementing a potential field often lead to the robot to touch the obstacle while trying to avoid it. To prevent this, the robot should **probe** for the obstacle **tangeant**, then **nudge one robot length**. It should repeat these two steps until the nudge step intersect with the original path.

<p align="center">
    <img src="images/local-nav-avoidance-steps.png" alt="avoidance-steps" width="400"/>
</p>

This mechanism is implemented using a **state machine**. Two states are added to initially exit the path, in order to **avoid detecting a false intersection** with the path from the very start.

<p align="center">
    <img src="images/local-nav-avoidance-state-machine.png" alt="state-machine" width="300"/>
</p>

### Collision detection with the path

Knowing the **path the robot must follow** (blue), and the **next nudge step segment** (green), we can compute when and where the global path and the avoidance trajectory will **intersect** :

```python
def cross2d(a: np.ndarray, b: np.ndarray):
    return a[0] * b[1] - a[1] * b[0]

def trajectory_direction(path: np.ndarray) -> int:
    if path.shape[0] >= 3:
        return -1 if cross2d(path[1] - path[0], path[2] - path[1]) < 0 else 1
    else:
        return 1

def segments_intersection_point(s1: np.ndarray, s2: np.ndarray) -> np.ndarray | None:
    r = s1[1] - s1[0]
    s = s2[1] - s2[0]
    r_cross_s = cross2d(r, s)
    if abs(r_cross_s) < 1e-9:
        return None

    diff = s2[0] - s1[0]
    t = cross2d(diff, s) / r_cross_s
    u = cross2d(diff, r) / r_cross_s

    if 0 <= t <= 1 and 0 <= u <= 1:
        return s1[0] + t * r

    return None

def path_intersection_point(path: np.ndarray, segment: np.ndarray) -> tuple[np.ndarray | None, int | None]:
    for i in range(len(path) - 1):
        path_segment = path[i:(i + 2)]
        point = segments_intersection_point(segment, path_segment)
        if point is not None:
            return point, i
    return None, None
```

## GUI Dashboard and Demo

The project provides a single interactive dashboard that visualises planning, perception and filtering together. The GUI is implemented across three main files:

- `dashboard.py` — constructs the interactive 3-panel dashboard, overlays a live camera frame with world→pixel projections, and exposes plotting handles to update odometry/camera markers, EKF covariance traces and position errors.
- `globalnav_plot.py` — the mapping and plotting helper: computes the A* path (via `compute_global_path_with_debug`), draws original and inflated obstacles, optional visibility edges, and returns Matplotlib handles so the map can be updated live.
- `demo.py` — the main entry point used during experiments: starts the camera and Thymio threads (or runs in simulation), runs the EKF, feeds live measurements to the dashboard and writes logs.

What the dashboard shows (three panels):

- Left — Live camera image: A* path nodes and connecting path are projected onto the camera image using the inverse homography; odometry (blue) and camera detections (green) are shown as a small base circle plus a heading arrow and updated every sample.
- Middle — Global navigation map: original vs inflated obstacles, the computed A* path, and live trajectories/poses for odometry and camera detections; this plot is produced by `globalnav_plot.py` and updated in real time by the dashboard.
- Right — Filtering & performance: top—EKF covariance diagonals (σx², σy², σθ²) over time; bottom—position error (‖odom − est‖) so you can visually check filter performance and drift vs. camera corrections.

Running the system: start the full demo from the `Code/` folder with `python3 demo.py` (simulation and no-camera/no-thymio modes are supported for offline testing). The demo logs results to `Code/log.csv` for post-hoc analysis and the notebook contains small reproducible examples that use `compute_global_path_with_debug()` to reproduce and inspect planner behaviour.


## Putting all togheter

-choice of epsilon
